# MTtime workflows example

Use GeoNet event **2026p530771** to run a staged moment-tensor inversion and the
equivalent workflow through `run_auto_cmt()`. The example holds the epicentre
fixed and evaluates source depth at the catalogue origin before testing origin
shifts. The station-selection audit, leave-one-station-out results and depth and
shift curves show how those choices affect the solution.

Start with the layered PyFK example. It uses the IASP91 file supplied with ObsPy
unless you choose a local profile. The NZ velocity-grid, Syngine and prepared
SW4 routes are available through the backend setting. The optional correlation,
station removal and resampling examples are at the end.

Use the Jupyter kernel for your installed MTtime checkout. In your existing
environment, `python -m pip install --no-deps -e .` from the repo root
refreshes MTtime while keeping the installed dependencies. The PyFK route uses the
compiled backend on Linux or WSL. Syngine uses EarthScope's web service.
The map view can be added to `plot_views` when Cartopy is installed.

## 1. Configure the event, station selection and timing

In [ ]:
import os
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import obspy.taup

import mttime
from mttime.workflows import mttime_helpers as mt, run_auto_cmt
from mttime.workflows.gf_helpers import build_regular_velocity_grid

print(f"MTtime {mttime.__version__} · workflow {mt.__version__}")

station_columns = [
    "station_id", "channel_prefix", "distance_km", "azimuth_deg",
    "azimuth_sector", "used_components", "selection_status",
]

def show_jackknife(table):
    if table is None or table.empty:
        return
    columns = [
        "station_id", "opposing_station_id", "n_stations_remaining",
        "loo_depth_km", "loo_reference_time_shift_s", "delta_mw",
        "loo_total_vr_percent", "kagan_angle_deg", "heldout_vr_percent", "status", "reason",
    ]
    display(table.reindex(columns=columns).round(3))

def read_station_csv(path):
    """Load raw station metadata, preserving location codes and relative paths."""
    path = Path(path).expanduser().resolve()
    stations = pd.read_csv(path, dtype={"event_id": str, "station_id": str, "location": str},
                           keep_default_na=False)
    for column in ("waveform_path", "stationxml_path"):
        stations[column] = stations[column].map(
            lambda file: str((path.parent / Path(file).expanduser()).absolute())
        )
    return stations

In [ ]:
# The catalogue geometry and origin from the original notebook.
event_id = "2026p530771"
event_time = "2026-07-16T09:14:54.328610Z"
event_lon, event_lat = 167.681930541992, -45.038818359375
event_depth_km = 37.5175437927246
magnitude = 5.9                     # Controls the acquisition radius
event_args = (event_id, event_time, event_lon, event_lat, event_depth_km)

output_base = Path(os.environ.get("MTTIME_RUN_ROOT", Path.home() / "mttime_runs"))
event_root = (output_base / event_id).expanduser().resolve()
staged_root = event_root / "staged"

waveform_source = "fdsn"              # "fdsn" or "local"
local_station_csv = None              # A matching metadata/stations_downloaded.csv
auto_waveform_source = "local"       # "local" reuses section 3, "fdsn" acquires waveforms
acquisition = dict(
    client="GEONET",                   # Also accepts an ObsPy FDSN Client instance
    min_radius_km=0.0,
    max_radius_km=None,               # Magnitude 5.9 resolves to about 601 km
    radius_scale_factor=1.66,
    ground_level=True,
    channels=("HH?", "BH?", "LH?"),
    channel_priority=("HH", "BH", "LH"),
    taup_model="iasp91",
    before_p=60.0,
    after_s=300.0,
)

azimuth_control = {
    "azimuth_selection": True,
    "azimuth_min_sectors": 3,
    "azimuth_max_stations_per_sector": 2,
    "minimum_stations": 4,
}
station_jackknife = {
    "jackknife_min_stations": 4,
    "jackknife_opposing_station": False,
    "jackknife_opposition_tolerance_deg": 20.0,
}
drop_stations = []                   # For example, ["NZ.EAZ.10"]

`azimuth_control=True` is shorthand for the four values above. Eight
45° sectors begin at north. After manual exclusions, the selector ranks
active stations within a sector by channel priority, distance and station ID.
The processed table keeps each exact network, station and location code.

The default jackknife holds out one active station at a time and refits the
existing candidates. `jackknife_min_stations` counts the stations that must
**remain**. Turning on `jackknife_opposing_station` tests omissions of two
stations at nearly opposite azimuths. The tolerance applies to that option.

The magnitude of 5.9 is the input used to size the acquisition radius for
this example. Keep the geometry and origin together when using a saved
station CSV or external Green's functions.

In [ ]:
# The full grid from the earlier notebook was 27.5 to 77.5 km in 1 km steps.
depth_bounds_km = (27.5, 77.5)
depth_step_km = 5.0                    # A coarse first pass for the example
depths_km = np.round(np.arange(
    depth_bounds_km[0], depth_bounds_km[1] + 1e-8, depth_step_km,
), 4)
reference_time_shifts = np.arange(-5.0, 6.0, 1.0)
run_correlation_examples = False
residual_correlation_margin_s = 5.0

waveform = dict(motion="DISP", freqmin=0.02, freqmax=0.05, dt=1.0)
processing = dict(
    corners=3, water_level=None,
    window_start_phase="P", time_before_window_start=30.0,
    window_end_phase="S", time_after_window_end=250.0,
    inversion_duration=None,          # Keep a station-specific inversion duration
    max_reference_time_shift=float(np.abs(reference_time_shifts).max()),
    max_correlation_shift=residual_correlation_margin_s if run_correlation_examples else 0.0,
    pad_missing_prearrival=True,
    reduction_velocity_km_s=0.0,
)
inversion = dict(
    components="ZRT", weight="variance",
    drop_stations=drop_stations, azimuth_control=azimuth_control,
    station_jackknife=station_jackknife,
)
output_options = dict(show=False, figure_format="png", write_option="preferred")
solution_columns = [
    "depth_km", "reference_time_shift", "mw", "total_vr_percent",
    "dc_percent", "clvd_percent", "iso_percent",
]

# Optional analyses in section 5.
min_preferred_station_vr = None        # Try 10.0 if enough stations will remain
run_resampling_example = False

Decrease `depth_step_km` to 1.0 for the original depth grid. Each additional
depth requires a GF set for every receiver. The reference-time margin is
calculated from the largest absolute shift. If the correlation examples are
enabled, processing also reserves a separate margin for individual stations.

If the best fit lies at either end of the depth grid, report it as a boundary solution. Any further depth search requires GFs for the added depths.

## 2. Choose one Green's-function library

| `gf_backend` | Inputs |
|---|---|
| `pyfk_1d` | A layered profile. The default reads ObsPy's bundled IASP91 table. |
| `pyfk_profile` | One vertical profile at the event from the tabular NZ model. |
| `pyfk_path` | One mean or median layered profile along each source–station path. |
| `pyfk_station` | One vertical profile beneath each station. |
| `syngine` | An EarthScope model such as `iasp91_2s` or `prem_a_2s`. |
| `custom` | A completed MTtime SAC library and manifest, such as the output of `sw4_greens.export_mttime_gfs()`. |

The three NZ model routes reduce 3D material properties to layered PyFK
profiles. For a full 3D SW4 calculation use the custom route after export.
Set the same source coordinates, depth, motion, filter, sampling and window
used for the export. A fixed hypocentre run uses one manifest depth and
`reference_time_shifts = np.array([0.0])`. The staged and automated
runs keep separate outputs for each backend.

In [ ]:
gf_backend = "pyfk_1d"
vm_dir = Path(os.environ.get("MTTIME_VM_DIR", "vm")).expanduser()
profile_csv = None               # Optional Path; columns depth, Vp and Vs
syngine_model = "iasp91_2s"
custom_manifest = None           # For example, Path("path/to/sw4_gf_manifest.csv")
custom_green_dir = None          # Set if the manifest has no usable path column
profile_options = {}             # For example, {"depth_col": "Depth_km", "vp_col": "VP", "vs_col": "VS"}

pyfk_options = dict(
    pyfk_dt=1.0, pyfk_npt=None, samples_before_first_arrival=30,
    pyfk_native_motion="VEL", batch_common_model=True,
    pyfk_config_kwargs={"dk": 0.125, "kmax": 15.0},
)
velocity_model = None
if gf_backend == "pyfk_1d":
    if profile_csv is None:
        model_file = Path(obspy.taup.__file__).parent / "data" / "iasp91.tvel"
        velocity_model = pd.read_csv(
            model_file, sep=r"\s+", skiprows=2, names=["depth", "Vp", "Vs", "density"],
        )
        # Keep the regional crust and upper mantle and a terminal half-space.
        velocity_model = velocity_model.loc[velocity_model["depth"] <= 210.0].copy()
    else:
        velocity_model = pd.read_csv(Path(profile_csv).expanduser())
    gf_source = "pyfk"
    gf_options = {**pyfk_options, **profile_options}
elif gf_backend in {"pyfk_profile", "pyfk_path", "pyfk_station"}:
    node_cols = ["x(km)", "y(km)", "Depth(km_BSL)", "Latitude", "Longitude"]
    nz = pd.read_csv(
        vm_dir / "vlnzw2p3dnxyzltln.tbl.txt", sep=r"\s+", skiprows=2,
        names=["Vp", "Vp/Vs", "Vs", "Density", "SF_Vp", "SF_Vp/Vs", *node_cols],
    )
    for phase in ("p", "s"):
        q_name = f"Q{phase}"
        q_nodes = pd.read_csv(
            vm_dir / f"{q_name}nzw2p3xyzltln.tbl.txt", sep=r"\s+", skiprows=2,
            names=[q_name, f"SF_{q_name}", *node_cols],
        )
        nz = nz.merge(q_nodes[[*node_cols, q_name]], on=node_cols, validate="one_to_one", how="left")
    velocity_model = build_regular_velocity_grid(
        nz, x_col="Longitude", y_col="Latitude", depth_col="Depth(km_BSL)",
        vp_col="Vp", vs_col="Vs", density_col="Density", qs_col="Qs", qp_col="Qp",
        coordinate_crs="EPSG:4326", interpolation_crs="EPSG:2193",
    )
    grid_mode = gf_backend.split("_", 1)[1]
    gf_source = "pyfk"
    gf_options = dict(
        pyfk_options, grid_mode=grid_mode, profile_location=(event_lon, event_lat),
        profile_crs="EPSG:4326", path_spacing_km=2.0, path_profile="mean",
        surface_depth_km=0.0, pyfk_flattening=True,
    )
elif gf_backend == "syngine":
    gf_source = "syngine"
    gf_options = dict(syngine_model=syngine_model, syngine_dt=None, max_workers=5)
elif gf_backend == "custom":
    if custom_manifest is None:
        raise ValueError("Set custom_manifest to the prepared MTtime GF manifest.")
    gf_source = "custom"
    gf_options = dict(manifest=custom_manifest, path_to_green=custom_green_dir)
    custom_receivers = set(
        custom_manifest["station_id"].astype(str) if isinstance(custom_manifest, pd.DataFrame)
        else pd.read_csv(Path(custom_manifest).expanduser(), usecols=["station_id"], dtype=str)["station_id"]
    )
else:
    raise ValueError(f"Unknown gf_backend {gf_backend!r}.")

backend_name = f"syngine_{syngine_model}" if gf_source == "syngine" else gf_backend
gf_root = staged_root / "libraries" / backend_name
fit_root = staged_root / "fits" / backend_name
auto_root = event_root / "auto" / backend_name
trial_depths = None if gf_source == "custom" else depths_km

`profile_options` can map a local CSV's columns to PyFK, for example
`dict(depth_col="Depth_km", vp_col="VP", vs_col="VS", density_col="Density")`.
The layered profile uses km, km/s and g/cm³. The NZ example matches the
attenuation rows to the velocity grid by their node coordinates. SAC files and manifests
for each backend have their own directory.

## 3. Run the staged workflow

### 3.1 Acquire waveforms and prepare observed SAC

In [ ]:
if waveform_source == "fdsn":
    event_df, raw_stations, download_log = mt.get_mseed_stationxml(
        *event_args, magnitude=magnitude, output_dir=staged_root,
        overwrite=False, **acquisition,
    )
elif waveform_source == "local":
    if local_station_csv is None:
        raise ValueError("Set local_station_csv to a saved station table for this event.")
    raw_stations = read_station_csv(local_station_csv)
    if "event_id" in raw_stations and not raw_stations["event_id"].eq(event_id).all():
        raise ValueError("The local station table belongs to a different event.")
    if "origin_time" in raw_stations:
        saved_origins = pd.to_datetime(raw_stations["origin_time"], utc=True)
        if not (saved_origins - pd.Timestamp(event_time)).abs().le(pd.Timedelta(microseconds=1)).all():
            raise ValueError("The local station table has a different origin. Update its geometry and arrivals.")
    for key, value in (("event_lon", event_lon), ("event_lat", event_lat), ("event_depth_km", event_depth_km)):
        if key in raw_stations and not np.allclose(raw_stations[key], value, atol=1e-6, rtol=0.0):
            raise ValueError(f"The local station table has a different {key}.")
    event_df = pd.DataFrame([dict(
        event_id=event_id, origin_time=event_time, event_lon=event_lon,
        event_lat=event_lat, event_depth_km=event_depth_km, magnitude=magnitude,
    )])
    download_log = pd.DataFrame()
else:
    raise ValueError("Use waveform_source='fdsn' or 'local'.")

if gf_source == "custom" and not raw_stations.empty:
    raw_stations = raw_stations.loc[raw_stations["station_id"].isin(custom_receivers)].copy()
if not download_log.empty:
    display(download_log["status"].value_counts().rename("stations").to_frame())
if raw_stations.empty:
    raise ValueError("No usable waveform records. Review the station filters and download log.")

_, processed_stations, processing_log = mt.process_mseed_sac(
    event_df, raw_stations, output_dir=staged_root,
    overwrite=False, **waveform, **processing,
)
display(processing_log["status"].value_counts().rename("stations").to_frame())
if processed_stations.empty:
    raise ValueError(f"Review {staged_root / 'metadata' / 'processing_log.csv'}.")
display(processed_stations[
    ["station_id", "channel_prefix", "distance_km", "azimuth_deg", "npts", "dt"]
].round(2))

### 3.2 Build or load the library

The staged generators retain all usable observations so you can change
station selection and repeat the inversion using the same files. The
automated workflow selects its network before calculating new GFs.
For custom files, section 4 is also a standalone route that validates
the manifest against the requested processing and geometry.

In [ ]:
model_manifest = None
if gf_source == "pyfk":
    gf_manifest, model_manifest = mt.generate_pyfk_gfs(
        event_df, processed_stations, velocity_model,
        output_dir=gf_root, depths_km=depths_km, overwrite=False, **gf_options,
    )
    green_dir = gf_root / "greens"
elif gf_source == "syngine":
    gf_manifest = mt.generate_syngine_gfs(
        event_df, processed_stations,
        output_dir=gf_root, depths_km=depths_km, overwrite=False, **gf_options,
    )
    green_dir = gf_root / "greens"
else:
    if isinstance(custom_manifest, pd.DataFrame):
        gf_manifest = custom_manifest.copy()
        manifest_dir = Path.cwd()
    else:
        manifest_file = Path(custom_manifest).expanduser().resolve()
        gf_manifest = pd.read_csv(manifest_file, dtype={"station_id": str})
        manifest_dir = manifest_file.parent
    if custom_green_dir is None:
        parents = {(manifest_dir / Path(file).expanduser()).parent.resolve() for file in gf_manifest["path"]}
        if len(parents) != 1:
            raise ValueError("Set custom_green_dir to the directory containing the MTtime SAC files.")
        green_dir = parents.pop()
    else:
        green_dir = (manifest_dir / Path(custom_green_dir).expanduser()).resolve()
data_dir = staged_root / "processed" / "data"
if gf_manifest.empty:
    raise ValueError("The Green's-function library returned an empty manifest.")
depth_column = "requested_depth_km" if "requested_depth_km" in gf_manifest else "source_depth_km"
tested_depths = sorted(gf_manifest[depth_column].astype(float).unique()) if trial_depths is None else trial_depths
print(f"{len(gf_manifest):,} GF traces · {len(tested_depths)} depths · {green_dir}")
display(gf_manifest.filter(items=["station_id", depth_column, "basis", "npts", "dt_s"]).head())

### 3.3 Compare the five- and six-component tensor inversions

Degree 5 fits a deviatoric moment tensor. Degree 6 also fits the isotropic
component. Both fits start at zero global shift and use the same settings.
Inspect the selection audit and the station omissions before interpreting
the best variance reduction.

In [ ]:
fixed_results = {}
for degree in (5, 6):
    # The solution tables and saved figures retain the full depth search.
    with redirect_stdout(StringIO()):
        stations_used, tdmt, solutions, station_fits, products = mt.run_mttime_inversion(
            event_df, processed_stations, gf_manifest,
            output_file=fit_root / f"degree_{degree}" / "mtinv.in",
            path_to_data=data_dir, path_to_green=green_dir, depths_km=trial_depths,
            degree=degree, correlate=False, reference_time_shift=0.0,
            plot_views=("waveform", "depth", "cmt") if degree == 5 else ("lune", "cmt"),
            **inversion, **output_options,
        )
    fixed_results[degree] = (tdmt, solutions, station_fits, products)

preferred_fixed = pd.concat([
    item[1].loc[item[1]["preferred"], ["degree", *solution_columns]]
    for item in fixed_results.values()
], ignore_index=True)
display(preferred_fixed.round(3))

tdmt, solutions, station_fits, products = fixed_results[5]
preferred = solutions.loc[solutions["preferred"]].iloc[0]
if tdmt.station_selection is not None:
    display(tdmt.station_selection[station_columns].round(2))
show_jackknife(tdmt.station_jackknife)
if solutions["depth_km"].nunique() > 1:
    bounds = solutions["depth_km"].agg(["min", "max"])
    if preferred["depth_km"] in bounds.values:
        print(f"Preferred degree-5 depth is on the tested boundary ({preferred['depth_km']:g} km).")

fig, ax = plt.subplots(figsize=(6.8, 3.5), layout="constrained")
for degree, (_, depths, _, _) in fixed_results.items():
    ax.plot(depths["depth_km"], depths["total_vr_percent"], "o-", ms=4, label=f"Degree {degree}")
ax.set(xlabel="Source depth (km)", ylabel="Variance reduction (%)")
ax.grid(alpha=0.22)
ax.legend(frameon=False)
plt.show()

### 3.4 Scan origin time at the preferred depth

A positive shift moves the source reference later than the catalogue origin.
The same azimuth-selected network is used for every shift. The jackknife in
this section can choose a different shift for each omission. It uses the one
depth selected by the degree-5 fit above. Pass the full depth list to test
depth and shift jointly.

In [ ]:
scan_depths = [float(preferred["depth_km"])]
scan_options = dict(
    reference_time_shifts=reference_time_shifts,
    path_to_data=data_dir, path_to_green=green_dir,
    depths_km=scan_depths, degree=5, plot_views=(),
    progress=True, **inversion, **output_options,
)
shift_summary, shift_solutions, shift_station_fits, shift_products = mt.mttime_inversion_oshift(
    event_df, processed_stations, gf_manifest,
    output_dir=fit_root / "origin_shifts", correlate=False, **scan_options,
)
display(shift_summary[[*solution_columns, "best_vr_across_shifts"]].round(3))
records = shift_summary.attrs["station_jackknife"]
show_jackknife(None if records is None else pd.DataFrame.from_records(records))

fig, ax = plt.subplots(figsize=(6.8, 3.5), layout="constrained")
ax.plot(shift_summary["reference_time_shift"], shift_summary["total_vr_percent"], "o-", ms=4)
ax.set(xlabel="Reference-time shift (s)", ylabel="Variance reduction (%)",
       title=f"Degree 5 · {scan_depths[0]:g} km")
ax.grid(alpha=0.22)
plt.show()

## 4. Run the same event with `run_auto_cmt`

The default `auto_waveform_source="local"` passes the raw station table
from section 3. The wrapper processes those records, prepares the Green's
functions and fits the degree-5 depth grid at the input origin in one call.
It also runs the station controls above and writes a CMT plot. For a fresh
end-to-end run, set `auto_waveform_source="fdsn"` and run this section after
sections 1 and 2. It will acquire the GeoNet records itself.

Set `auto_reference_time_shifts = reference_time_shifts` for a joint
depth/time search. To reproduce the staged shift-only search, also set
`auto_depths = scan_depths`. You can start offline with `local_station_csv`
in section 1. For a custom library, use the corresponding local station
table and waveform settings from the export.

The comparison uses the same processing and station controls. The staged PyFK library includes
all usable receivers. The automated workflow calculates GFs for its selected receivers.
Different receiver batches and internal GF lengths can produce small numerical differences.

In [ ]:
auto_reference_time_shifts = None      # Set to reference_time_shifts for a scan
auto_depths = trial_depths
if auto_waveform_source == "local":
    auto_stations = globals().get("raw_stations", local_station_csv)
    if auto_stations is None:
        raise ValueError("Run section 3.1 or set local_station_csv in section 1.")
    if gf_source == "custom" and not isinstance(auto_stations, pd.DataFrame):
        auto_stations = read_station_csv(auto_stations)
    if gf_source == "custom":
        auto_stations = auto_stations.loc[auto_stations["station_id"].isin(custom_receivers)].copy()
    auto_station_input = {"station_df": auto_stations}
else:
    auto_station_input = {"client": acquisition["client"]}
auto_acquisition = {name: value for name, value in acquisition.items() if name != "client"}
auto_inversion_options = dict(write_option="preferred", plot_views=("waveform", "depth", "cmt"))
if auto_reference_time_shifts is not None:
    auto_inversion_options.update(progress=True, plot_views=())

with redirect_stdout(StringIO()):
    auto = run_auto_cmt(
        *event_args, magnitude, output_dir=auto_root, waveform_source=auto_waveform_source,
        gf_source=gf_source, velocity_model=velocity_model, gf_options=gf_options,
        depths_km=auto_depths, reference_time_shifts=auto_reference_time_shifts,
        processing_options=processing, degree=5, correlate=False, plot=True, show=False,
        inversion_options=auto_inversion_options,
        **auto_station_input, **auto_acquisition, **waveform, **inversion,
    )
display(auto["results"]["summary"][solution_columns].round(3))
display(auto["station_selection"][station_columns].round(2))
show_jackknife(auto["station_jackknife"])
display(auto["output_manifest"].query("status == 'written'")[["view", "path"]])
if "preferred_fixed" in globals() and auto_reference_time_shifts is None and auto_depths is trial_depths:
    comparison = pd.concat({
        "Staged": preferred_fixed.loc[preferred_fixed["degree"] == 5, solution_columns].reset_index(drop=True),
        "Automated": auto["results"]["summary"][solution_columns].reset_index(drop=True),
    })
    display(comparison.round(4))
    staged_ids = set(tdmt.mtinv_station_table["station"])
    auto_ids = set(auto["mtinv_stations"]["station"])
    print("Same stations in the staged and automated fits:", staged_ids == auto_ids)

The automated station selection is in `auto["station_selection"]`. Its
files live under `auto_root`, including `metadata/station_selection.csv`
and `results/station_jackknife.csv` for the fixed-origin call, or
`origin_shifts/station_jackknife.csv` for a shift scan. Choose `auto_root`
under an event-specific directory to reuse your raw and GF caches on reruns.

## 5. Optional sensitivity tests

These examples use the observations and GFs from section 3.

### 5.1 Manual exclusion based on the preferred solution

Apply the cutoff once to the station fits at the preferred depth. This
shows the effect of removing low-fit records selected from that solution.
Compare the tensor and retained station set as well as VR.

In [ ]:
if min_preferred_station_vr is not None:
    preferred_fits = station_fits.loc[station_fits["preferred"]]
    low_vr_ids = preferred_fits.loc[
        preferred_fits["VR"].lt(min_preferred_station_vr), "station"
    ].unique().tolist()
    if low_vr_ids:
        sensitivity = dict(inversion)
        sensitivity["drop_stations"] = sorted(set(drop_stations) | set(low_vr_ids))
        with redirect_stdout(StringIO()):
            _, low_vr_run, low_vr_solutions, _, _ = mt.run_mttime_inversion(
                event_df, processed_stations, gf_manifest,
                output_file=fit_root / "low_station_vr" / "mtinv.in",
                path_to_data=data_dir, path_to_green=green_dir, depths_km=trial_depths,
                degree=5, correlate=False, plot_views=(), **sensitivity, **output_options,
            )
        print("Excluded:", low_vr_ids)
        display(low_vr_solutions.loc[low_vr_solutions["preferred"], solution_columns].round(3))
    else:
        print("No active station fell below the chosen cutoff.")

### 5.2 Signed and absolute correlation

Set `run_correlation_examples=True` in section 1 and rerun processing
before running this cell. Both examples use the same origin-shift grid and
allow a bounded residual correction around each global shift. Signed
correlation maximises the correlation coefficient. Absolute correlation
maximises its magnitude.

In [ ]:
if run_correlation_examples:
    scans = {"No residual correlation": shift_summary}
    for mode, abs_max in (("Signed", False), ("Absolute", True)):
        scans[mode], _, _, _ = mt.mttime_inversion_oshift(
            event_df, processed_stations, gf_manifest,
            output_dir=fit_root / f"correlation_{mode.lower()}",
            correlate=True, correlation_abs_max=abs_max, **scan_options,
        )
    display(pd.concat({
        mode: table.loc[table["best_vr_across_shifts"], solution_columns].reset_index(drop=True)
        for mode, table in scans.items()
    }).round(3))

### 5.3 Opposing-pair resampling

This optional ensemble samples multiple opposing-pair deletion designs.
The main jackknife above uses a deterministic leave-one-station-out
test by default. Here the ensemble uses all processed observations after
manual exclusions. If your station network has few opposing pairs,
reduce the number of unique realizations or widen the opposition tolerance.

In [ ]:
if run_resampling_example:
    resample_options = dict(
        loo_pairs=1, opposition_tolerance_deg=45.0, min_remaining_stations=4,
        max_relative_radial_difference=None, max_azimuth_gap_increase_deg=30.0,
    )
    audit = tdmt.station_selection
    omitted_ids = (
        drop_stations if audit is None
        else audit.loc[audit["selection_status"].eq("manual_drop"), "station_id"]
    )
    kept = processed_stations.loc[~processed_stations["station_id"].isin(omitted_ids)]
    _, pairs = mt.select_opposing_station_pairs(kept, **resample_options, random_state=91)
    display(pairs)
    ensemble, realizations, fits, pairs = mt.run_mttime_resampling(
        event_df, processed_stations, gf_manifest,
        output_dir=fit_root / "opposing_pair_resampling",
        path_to_data=data_dir, path_to_green=green_dir,
        depths_km=trial_depths, components="ZRT", degree=5, weight="variance",
        base_drop_stations=drop_stations, n_realizations=5, seed=91,
        unique_designs=True, **resample_options,
    )
    display(ensemble[["representative_type", "depth_km", "depth_unc_km", "mw", "mw_unc",
                      "total_vr_percent", "vr_unc_percent", "tensor_angle_q84_deg"]].round(3))